In [ ]:
import pandas as pd
import numpy as numpy
import ast # [abstract syntax tree]

In [ ]:
#Reading CSVs

credits = pd.read_csv('tmdb_5000_credits.csv')
movies = pd.read_csv('tmdb_5000_movies.csv')

In [ ]:
credits.head()

In [ ]:
movies.head()

In [ ]:
# Merging 

movies = movies.merge(credits, left_on ='title', right_on = 'title')

In [ ]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [ ]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies['genres']

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
movies['cast'][0]

In [ ]:
# movies['cast'] = movies['cast'].apply(lambda x: [i['name'] for i in ast.literal_eval(x)[:3]]) # Only Top 3 Actors

In [ ]:
def fetch_top_3_cast(obj):
    L = []
    counter = 0
    # try-except lagaya hai taaki agar koi missing value (NaN) aaye toh code crash na ho
    try:
        for i in ast.literal_eval(obj):
            if counter != 3:
                L.append(i['name'])
                counter += 1
            else:
                break
        return L
    except (ValueError, TypeError):
        return [] # Agar data me problem hai, toh khali list return karega

# Function apply kijiye
movies['cast'] = movies['cast'].apply(fetch_top_3_cast)

In [ ]:
movies['crew'][0]

In [ ]:
movies['crew'] = movies['crew'].apply(lambda x: [i['name'] for i in ast.literal_eval(x) if i['job'] == 'Director']) # Only Top 3 Actors

In [ ]:
movies['tags'] = movies['genres'] + movies['keywords'] + movies['crew']

In [ ]:
movies['tags']

In [ ]:
movies = movies[['movie_id', 'title', 'overview', 'tags']]

In [ ]:
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

In [ ]:
movies['tags']

In [ ]:
movies.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['tags'])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [ ]:
def get_recommendations(title, cosine_sim = cosine_sim): # Defines a function that takes a movie title and similarity matrix
    idx = movies[movies['title'] == title].index[0] # Finds the dataframe index of the requested movie
    sim_scores = list(enumerate(cosine_sim[idx])) # Gets similarity scores for this movie and attaches an index to each score
    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True) # Sorts the scores from highest to lowest similarity
    sim_scores = sim_scores[1:11] # Gets the Top 10 similar movies (skips index 0 because that's the same movie)
    movie_indices = [i[0] for i in sim_scores] # Extracts only the indices from the top 10 list
    return movies['title'].iloc[movie_indices] # Fetches and returns the actual movie titles using those indices

In [ ]:
print(get_recommendations('Iron Man'))

In [ ]:
# `import pickle` allows you to save complex, pre-calculated 
# Python data objects (like your trained machine learning model) 
# into a file, so your web app can instantly load them later without 
# having to recalculate everything from scratch.

import pickle
with open('movie_data.pkl', 'wb') as file:
    pickle.dump((movies, cosine_sim), file)

In [ ]:
import numpy as np

cosine_sim = cosine_sim.astype(np.float32)

import pickle
pickle.dump(movies.to_dict(), open('movie_dict.pkl', 'wb'))
pickle.dump(cosine_sim, open('similarity.pkl', 'wb'))